<a href="https://colab.research.google.com/github/PALAK7890/Time_value/blob/main/Time_Value.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [73]:
import pandas as pd
import numpy as np
events = pd.read_csv("startup_realistic_event_log.csv")
users = pd.read_csv("startup_realistic_user_summary.csv")

In [57]:
events["event_time"] = pd.to_datetime(events["event_time"])
users["signup_time"] = pd.to_datetime(users["signup_time"])
users["first_value_time"] = pd.to_datetime(users["first_value_time"])
users["last_active_time"] = pd.to_datetime(users["last_active_time"])

In [58]:
events = events.drop_duplicates()
users = users.drop_duplicates()

In [97]:
events["event_name"] = events["event_name"].str.lower().str.strip()
events["event_category"] = events["event_category"].str.lower().str.strip()
events["event_time"] = pd.to_datetime(events["event_time"], errors="coerce")

events = events.dropna(subset=["event_time"])

events["event_date"] = events["event_time"].dt.date

In [88]:
users["ttv_clean"] = users["time_to_value_hours"].fillna(999)

def ttv_bucket(x):
    if x == 999:
        return "Never Reached Value"
    elif x <= 1:
        return "Under 1 Hour"
    elif x <= 24:
        return "1-24 Hours"
    elif x <= 72:
        return "1-3 Days"
    elif x <= 168:
        return "3-7 Days"
    else:
        return "7+ Days"

users["ttv_bucket_custom"] = users["ttv_clean"].apply(ttv_bucket)

users["activation_velocity_index"] = np.where(
    users["reached_value"] == True,
    1 / (1 + users["ttv_clean"]),
    0
)

In [90]:
users["engagement_score"] = (
    users["sessions"] * 0.4 +
    users["feature_breadth"] * 0.3 +
    users["stickiness_ratio"] * 0.3
)
value_events = [
    "create_project",
    "connect_integration",
    "invite_team_member",
    "export_report",
    "publish_dashboard"
]

value_depth = (
    events[events["event_name"].isin(value_events)]
    .groupby("user_id")["event_name"]
    .nunique()
    .reset_index(name="value_depth_score")
)

users = users.merge(value_depth, on="user_id", how="left")
users["value_depth_score"] = users["value_depth_score"].fillna(0)


In [95]:
# ===============================
# 6. FRICTION RECOVERY SCORE - FIXED
# ===============================

# Remove old friction columns if they exist
users = users.drop(
    columns=["first_friction_time", "post_friction_activity", "friction_recovery_score"],
    errors="ignore"
)

friction_only = events[events["event_category"] == "friction"].copy()

first_friction = (
    friction_only
    .groupby("user_id", as_index=False)["event_time"]
    .min()
    .rename(columns={"event_time": "first_friction_time"})
)

users = users.merge(first_friction, on="user_id", how="left")

events_temp = events.merge(
    users[["user_id", "first_friction_time"]],
    on="user_id",
    how="left"
)

post_friction = events_temp[
    events_temp["first_friction_time"].notna() &
    (events_temp["event_time"] > events_temp["first_friction_time"])
]

post_friction_activity = (
    post_friction
    .groupby("user_id")
    .size()
    .reset_index(name="post_friction_activity")
)

users = users.merge(post_friction_activity, on="user_id", how="left")
users["post_friction_activity"] = users["post_friction_activity"].fillna(0)

users["friction_recovery_score"] = np.where(
    users["friction_events"] > 0,
    users["post_friction_activity"] / users["friction_events"],
    0
)

users["friction_recovery_score"] = users["friction_recovery_score"].clip(0, 1)

print(users[["user_id", "first_friction_time", "post_friction_activity", "friction_recovery_score"]].head())

   user_id            first_friction_time  post_friction_activity  \
0  U000001  2025-11-22 16:13:39.000000000                     2.0   
1  U000002                            NaN                     0.0   
2  U000003                            NaN                     0.0   
3  U000004                            NaN                     0.0   
4  U000005  2026-01-14 18:50:25.000000000                    45.0   

   friction_recovery_score  
0                      1.0  
1                      0.0  
2                      0.0  
3                      0.0  
4                      1.0  


In [98]:
# ===============================
# 7. HABIT FORMATION SCORE - FIXED
# ===============================

users = users.drop(
    columns=["active_days_count", "habit_formation_score"],
    errors="ignore"
)

events["event_date"] = events["event_time"].dt.date

daily_activity = (
    events.groupby(["user_id", "event_date"])
    .size()
    .reset_index(name="daily_events")
)

active_days_count = (
    daily_activity.groupby("user_id")["event_date"]
    .nunique()
    .reset_index(name="active_days_count")
)

users = users.merge(active_days_count, on="user_id", how="left")

users["active_days_count"] = users["active_days_count"].fillna(0)

users["habit_formation_score"] = (
    users["active_days_count"] / users["observed_lifetime_days"].replace(0, np.nan)
).fillna(0).clip(0, 1)

print(users[["user_id", "active_days_count", "observed_lifetime_days", "habit_formation_score"]].head())

   user_id  active_days_count  observed_lifetime_days  habit_formation_score
0  U000001                  4                      23               0.173913
1  U000002                 29                     260               0.111538
2  U000003                 29                     221               0.131222
3  U000004                 31                     153               0.202614
4  U000005                 31                     127               0.244094


In [102]:
# ===============================
# 8. FEATURE STICKINESS INDEX - FIXED
# ===============================

users = users.drop(
    columns=["sticky_features", "feature_stickiness_index"],
    errors="ignore"
)

# Make sure feature_name exists and is clean
events["feature_name"] = events["feature_name"].fillna("No Feature").astype(str).str.strip()

feature_events = events[events["feature_name"] != "No Feature"]

feature_usage = (
    feature_events.groupby(["user_id", "feature_name"])
    .size()
    .reset_index(name="feature_use_count")
)

sticky_features = (
    feature_usage[feature_usage["feature_use_count"] >= 3]
    .groupby("user_id")["feature_name"]
    .nunique()
    .reset_index(name="sticky_features")
)

users = users.merge(sticky_features, on="user_id", how="left")

users["sticky_features"] = users["sticky_features"].fillna(0)

users["feature_stickiness_index"] = (
    users["sticky_features"] / users["feature_breadth"].replace(0, np.nan)
).fillna(0).clip(0, 1)

print(users[["user_id", "sticky_features", "feature_breadth", "feature_stickiness_index"]].head())

   user_id  sticky_features  feature_breadth  feature_stickiness_index
0  U000001              0.0                2                  0.000000
1  U000002              6.0                6                  1.000000
2  U000003              6.0                3                  1.000000
3  U000004              7.0                9                  0.777778
4  U000005              6.0                9                  0.666667


In [103]:
events["event_time"] = pd.to_datetime(events["event_time"], errors="coerce")
events = events.dropna(subset=["event_time"])

latest_date = events["event_time"].max()

events["days_before_end"] = (latest_date - events["event_time"]).dt.days

events["decay_weight"] = np.exp(-events["days_before_end"] / 30)

decay_engagement = (
    events.groupby("user_id")["decay_weight"]
    .sum()
    .reset_index(name="time_decay_engagement_score")
)

users = users.drop(columns=["time_decay_engagement_score"], errors="ignore")

users = users.merge(decay_engagement, on="user_id", how="left")

users["time_decay_engagement_score"] = users["time_decay_engagement_score"].fillna(0)

max_decay = users["time_decay_engagement_score"].max()

users["time_decay_engagement_normalized"] = np.where(
    max_decay > 0,
    users["time_decay_engagement_score"] / max_decay,
    0
)

In [105]:
# Force-create time_decay_engagement_normalized safely

events["event_time"] = pd.to_datetime(events["event_time"], errors="coerce")
events = events.dropna(subset=["event_time"])

latest_date = events["event_time"].max()

events["days_before_end"] = (latest_date - events["event_time"]).dt.days
events["decay_weight"] = np.exp(-events["days_before_end"] / 30)

decay_engagement = (
    events.groupby("user_id")["decay_weight"]
    .sum()
    .reset_index(name="time_decay_engagement_score")
)

users = users.drop(
    columns=["time_decay_engagement_score", "time_decay_engagement_normalized"],
    errors="ignore"
)

users = users.merge(decay_engagement, on="user_id", how="left")
users["time_decay_engagement_score"] = users["time_decay_engagement_score"].fillna(0)

max_decay = users["time_decay_engagement_score"].max()

if max_decay > 0:
    users["time_decay_engagement_normalized"] = (
        users["time_decay_engagement_score"] / max_decay
    )
else:
    users["time_decay_engagement_normalized"] = 0

print(users[["time_decay_engagement_score", "time_decay_engagement_normalized"]].head())

users["engagement_intelligence"] = (
    users["habit_formation_score"] * 0.4 +
    users["feature_stickiness_index"] * 0.3 +
    users["time_decay_engagement_normalized"] * 0.3
).clip(0, 1)

   time_decay_engagement_score  time_decay_engagement_normalized
0                     0.051344                          0.001349
1                     8.944014                          0.234985
2                     4.406114                          0.115761
3                     7.049133                          0.185201
4                    12.625727                          0.331714


In [106]:
# Create final intelligence scores safely

users["activation_score"] = (
    users["activation_velocity_index"] * 0.5 +
    (users["value_depth_score"] / 5) * 0.5
).clip(0, 1)

users["engagement_intelligence"] = (
    users["habit_formation_score"] * 0.4 +
    users["feature_stickiness_index"] * 0.3 +
    users["time_decay_engagement_normalized"] * 0.3
).clip(0, 1)

users["friction_intelligence"] = (
    users["friction_rate"] * 0.5 +
    (1 - users["friction_recovery_score"]) * 0.5
).clip(0, 1)

users["user_health_score"] = (
    users["activation_score"] * 0.35 +
    users["engagement_intelligence"] * 0.40 +
    (1 - users["friction_intelligence"]) * 0.25
).clip(0, 1)
users["user_health_segment"] = pd.cut(
    users["user_health_score"],
    bins=[-0.01, 0.35, 0.65, 1.01],
    labels=["Critical", "At Risk", "Healthy"]
)

In [107]:

model_features = [
    "activation_score",
    "engagement_intelligence",
    "friction_intelligence",
    "monetization_readiness_score",
    "churn_early_warning_score",
    "early_48h_events",
    "feature_breadth",
    "friction_rate",
    "price_sensitivity_score",
    "onboarding_friction_score"
]

X = users[model_features].fillna(0)
y = users["churned"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=250,
    max_depth=6,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print("MODEL PERFORMANCE")
print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, proba))

MODEL PERFORMANCE
              precision    recall  f1-score   support

           0       0.77      0.95      0.85       436
           1       0.97      0.88      0.92       964

    accuracy                           0.90      1400
   macro avg       0.87      0.91      0.89      1400
weighted avg       0.91      0.90      0.90      1400

ROC-AUC: 0.9664623700940271


In [78]:
daily_activity = (
    events.groupby(["user_id", "event_date"])
    .size()
    .reset_index(name="daily_events")
)

active_days_count = (
    daily_activity.groupby("user_id")["event_date"]
    .nunique()
    .reset_index(name="active_days_count")
)

users = users.merge(active_days_count, on="user_id", how="left")
users["active_days_count"] = users["active_days_count"].fillna(0)

users["habit_formation_score"] = (
    users["active_days_count"] / users["observed_lifetime_days"]
).clip(0, 1)

In [108]:
importance = pd.DataFrame({
    "feature": model_features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("\nFEATURE IMPORTANCE")
print(importance)


FEATURE IMPORTANCE
                        feature  importance
1       engagement_intelligence    0.377234
7                 friction_rate    0.173267
3  monetization_readiness_score    0.133642
4     churn_early_warning_score    0.122914
0              activation_score    0.113234
2         friction_intelligence    0.048231
6               feature_breadth    0.015110
8       price_sensitivity_score    0.008046
9     onboarding_friction_score    0.005008
5              early_48h_events    0.003314


In [109]:

users["predicted_churn_probability"] = model.predict_proba(X)[:, 1]

users["predicted_churn_segment"] = pd.cut(
    users["predicted_churn_probability"],
    bins=[-0.01, 0.40, 0.70, 1.01],
    labels=[
        "Low Predicted Churn",
        "Medium Predicted Churn",
        "High Predicted Churn"
    ]
)

In [111]:
users["action_priority_score"] = (
    users["churn_early_warning_score"] * 0.5 +
    (1 - users["monetization_readiness_score"]) * 0.3 +
    users["friction_intelligence"] * 0.2
).clip(0, 1)

In [112]:
def action_segment(x):
    if x > 0.7:
        return "Immediate Intervention"
    elif x > 0.4:
        return "Monitor Closely"
    else:
        return "Healthy"

users["action_segment"] = users["action_priority_score"].apply(action_segment)

In [113]:
users["high_value_user"] = (
    users["monetization_readiness_score"] > 0.6
).astype(int)

users["high_value_at_risk"] = (
    (users["high_value_user"] == 1) &
    (users["churn_early_warning_score"] > 0.6)
).astype(int)

In [114]:
users.to_csv("final_killer_startup_churn_dataset.csv", index=False)
events.to_csv("cleaned_event_log.csv", index=False)
importance.to_csv("model_feature_importance.csv", index=False)

print("\nFILES SAVED SUCCESSFULLY")
print("final_killer_startup_churn_dataset.csv")
print("cleaned_event_log.csv")
print("model_feature_importance.csv")

print("\nFinal users shape:", users.shape)
print("Final events shape:", events.shape)


FILES SAVED SUCCESSFULLY
final_killer_startup_churn_dataset.csv
cleaned_event_log.csv
model_feature_importance.csv

Final users shape: (7000, 78)
Final events shape: (221549, 10)


In [115]:
final_cols = [
    # Identity
    "user_id",
    "signup_time",
    "acquisition_channel",
    "country",
    "persona_segment",

    # Core metrics
    "churned",
    "mrr",
    "reached_value",
    "time_to_value_hours",

    # Intelligence scores
    "activation_score",
    "engagement_intelligence",
    "friction_intelligence",
    "user_health_score",
    "monetization_readiness_score",
    "churn_early_warning_score",

    # Segments
    "user_health_segment",
    "retention_persona",
    "monetization_persona",

    # Action layer
    "action_priority_score",
    "action_segment",
    "high_value_at_risk",

    # Behavior
    "early_48h_events",
    "feature_breadth",
    "friction_rate",
    "stickiness_ratio"
]

dashboard_df = users[final_cols]

dashboard_df.to_csv("dashboard_ready_dataset.csv", index=False)

In [116]:
from google.colab import files

files.download("dashboard_ready_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [117]:
import zipfile

with zipfile.ZipFile("final_project_files.zip", "w") as z:
    z.write("dashboard_ready_dataset.csv")
    z.write("final_killer_startup_churn_dataset.csv")
    z.write("cleaned_event_log.csv")
    z.write("model_feature_importance.csv")

files.download("final_project_files.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>